# Local RAG over arXiv Physics Abstracts
## LangChain + SPECTER + Ollama (Gemini Judge)

**What this notebook demonstrates:**
- Why LangChain exists and what it adds over calling `ollama` directly
- The full Retrieval-Augmented Generation (RAG) pipeline over a real scientific corpus
- Concrete "before and after": same question, with and without retrieval
- Conversational memory so follow-up questions retain context
- LLM-as-judge: using Gemini with live web search to evaluate answer quality

**Corpus:** ~15–20k recent arXiv abstracts (submitted Jan 2024 onward) from two archives:
- **gr-qc** — General Relativity & Quantum Cosmology (loop quantum gravity, causal sets, canonical approaches)
- **hep-th** — High Energy Physics Theory (string theory, holography, AdS/CFT)

These two archives together cover the full landscape of quantum gravity research —
the centerpiece question for this demo.

**Prerequisites:**
- Ollama running locally (`ollama serve`) with `gemma3:4b` pulled
- SPECTER downloads automatically from HuggingFace Hub on first use (~440 MB)
- `GEMINI_API_KEY` set as an environment variable (for the judge step only)
- arXiv Kaggle snapshot JSON at its local path (see cell 2)
- Packages: `pip install langchain langchain-ollama langchain-community sentence-transformers faiss-cpu google-genai torch`

**Why LangChain over plain `import ollama`?**

Direct API calls are great for single-turn interactions, but LangChain provides:
- Composable **chains** (prompt → LLM → parser) with the `|` pipe operator (LCEL)
- A unified **retriever interface** for any vector store backend
- Built-in **memory** objects that manage conversation history automatically
- Swappable components — change `ChatOllama` to `ChatOpenAI` with one line

## 1. Environment Check

In [ ]:
import os
import torch
import ollama

LLM_MODEL     = "gemma3:4b"
SPECTER_MODEL = "sentence-transformers/allenai-specter"   # scientific paper similarity
GEMINI_MODEL  = "gemini-2.5-flash"
NUM_CTX       = 16384   # Ollama context window; default (~2048) silently truncates abstracts
RETRIEVE_K    = 12      # number of abstracts to retrieve per query

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

available = [m.model for m in ollama.list().models]
print("Available Ollama models:", available)

assert LLM_MODEL in available, f"Pull the LLM first: ollama pull {LLM_MODEL}"
print(f"\nLLM model     : {LLM_MODEL}  (context window: {NUM_CTX} tokens)")
print(f"Embed model   : {SPECTER_MODEL}")
print(f"               (downloads ~440 MB from HuggingFace on first use)")
print(f"Embed device  : {DEVICE}")
print(f"Retrieve k    : {RETRIEVE_K} papers per query")
print(f"Judge model   : {GEMINI_MODEL}  (requires GEMINI_API_KEY)")

gemini_key = os.environ.get("GEMINI_API_KEY", "")
if gemini_key:
    print("\n✓  GEMINI_API_KEY found — judge step (Section 8) will run.")
else:
    print("\n⚠  GEMINI_API_KEY not set — the judge step (Section 8) will be skipped.")
    print("   Set it with:  import os; os.environ['GEMINI_API_KEY'] = 'your-key'")

## 2. ChatOllama and the Pipe Operator

LangChain Expression Language (LCEL) uses `|` to chain components:

```
prompt | llm | parser
```

Each component has an `.invoke()` method and accepts the output of the previous step.
This is equivalent to nested function calls but reads like a data pipeline.

`ChatOllama` is a drop-in wrapper around the Ollama API — it takes the same model names
you'd pass to `ollama.chat()` but returns LangChain `AIMessage` objects that chain components
can work with.

In [ ]:
import textwrap
import time

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# num_ctx enlarges Ollama's context window beyond its ~2048-token default.
# Without this, retrieving 12 abstracts (~3-4k tokens) would be silently truncated
# before the model ever sees them — the single biggest reason RAG underperforms.
llm    = ChatOllama(model=LLM_MODEL, temperature=0.2, num_ctx=NUM_CTX)
parser = StrOutputParser()

simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}"),
])

simple_chain = simple_prompt | llm | parser

# Warm up the model (loads weights on first call)
print("Warming up model...")
t0 = time.perf_counter()
_ = simple_chain.invoke({"question": "hi"})
print(f"Model ready in {time.perf_counter() - t0:.1f}s")

## 3. The Problem with LLMs Alone (The "Without Retrieval" Baseline)

Before building RAG, let's see how the LLM answers our key question
**with no external knowledge** — using only what was baked into its weights during training.

Our question asks about the *current state* of a rapidly evolving research field.
Pay attention to:
- Is the answer **specific** — does it name researchers, papers, or recent results?
- Is it **current** — does it reflect post-2021 work, or a generic textbook overview?
- Is it **grounded** — can it cite any source, or is it synthesizing from vague memory?

We'll compare this answer to the RAG version in Section 7.

In [8]:
KEY_QUESTION = "What is the current state of quantum gravity research?"

print("Question:", KEY_QUESTION)
print("\n" + "=" * 65)
print("ANSWER WITHOUT RETRIEVAL (LLM training knowledge only):")
print("=" * 65)

t0 = time.perf_counter()
answer_no_rag = simple_chain.invoke({"question": KEY_QUESTION})
elapsed = time.perf_counter() - t0

print(textwrap.fill(answer_no_rag, width=80))
print(f"\n[{elapsed:.1f}s]")
print("\n⚠  Does the answer name specific papers, researchers, or post-2021 results?")
print("   We will compare this to the RAG answer in Section 7.")

Question: What is the current state of quantum gravity research?

ANSWER WITHOUT RETRIEVAL (LLM training knowledge only):
Quantum gravity research is active but lacks a definitive theory. Here’s a
concise overview:  *   **String Theory:** Remains the most prominent framework,
exploring gravity as arising from vibrating strings. *   **Loop Quantum
Gravity:** Focuses on quantizing spacetime itself, proposing it’s made of
discrete loops. *   **Other Approaches:** Include causal set theory, asymptotic
safety, and emergent gravity. *   **Experimental Challenges:** Direct
experimental verification is extremely difficult due to the extremely small
scales involved. Researchers are exploring potential indirect signatures like
subtle effects on cosmology or black holes.  Essentially, it’s a field of
ongoing theoretical investigation with no established consensus.

[3.2s]

⚠  Does the answer name specific papers, researchers, or post-2021 results?
   We will compare this to the RAG answer in Sect

## 4. Building the Knowledge Base

### Step 4a: The Corpus
We build a knowledge base from **~15–20k recent arXiv abstracts** (submitted Jan 2024 onward)
focused on the two archives where quantum gravity research actually lives:

| Archive | Topic | Approaches covered |
|---|---|---|
| `gr-qc` | General Relativity & Quantum Cosmology | Loop quantum gravity, causal sets, canonical quantization, quantum cosmology |
| `hep-th` | High Energy Physics — Theory | String theory, holography, AdS/CFT, black hole information |

Why these two? The "state of quantum gravity research" question spans **canonical/discrete**
approaches (gr-qc) *and* the **string/holographic** programme (hep-th) — using only one
archive would give a lopsided picture.

The corpus is loaded from a local **Kaggle arXiv metadata JSON snapshot** (streamed once
and cached to CSV for fast reloads). Papers are filtered by their **arXiv ID prefix** (true
submission date), not `update_date` — a 2003 paper revised in 2024 would pass a date filter
but is not "new" research.

### Step 4b: One Document Per Paper (No Splitting)
In a typical RAG demo we chunk long documents into overlapping windows.
Here we take a deliberate different approach: **each abstract is stored as a single `Document`**.

Why? An abstract is a self-contained atomic unit of meaning — every sentence describes
the *same* paper. Splitting it would fragment related ideas into separate chunks, making
retrieval return partial context that loses the paper's argument. When documents are
already short and coherent, one-chunk-per-document is the right strategy.

**Takeaway:** chunking strategy is data-dependent — know your documents before you split.

### Step 4c: SPECTER Embeddings → FAISS (Persisted)
Rather than a general-purpose embedding model, we use **SPECTER**
(`sentence-transformers/allenai-specter`) — a transformer fine-tuned on 146k *citing/cited*
paper pairs from Semantic Scholar (Cohan et al., 2020). The contrastive training signal
("papers that cite each other should be close; random pairs should be far") makes SPECTER
purpose-built for **paper-level similarity and retrieval**.

**Compare to SciBERT:** SciBERT is pretrained on scientific text and understands the domain
vocabulary, but uses mean-pooling of token embeddings without contrastive fine-tuning.
Its cosine similarities are less calibrated — retrieving *loosely* related papers rather
than the most semantically similar ones.  SPECTER fixes exactly that.

Because the corpus is ~15–20k papers (vs. the previous 600), embedding takes several minutes
on CPU.  The FAISS index is **persisted to disk** after the first build and reloaded
in seconds on every subsequent run.

In [ ]:
import os
import pandas as pd
from langchain_core.documents import Document

# ─── Configuration ────────────────────────────────────────────────────────────
ARXIV_PATH      = ('C:/Users/Graham West/Python Notebooks/'
                   'Meharry Teaching/Datasets/arXiv/arxiv-metadata-oai-snapshot.json')
ARXIV_CSV       = ('C:/Users/Graham West/Python Notebooks/'
                   'Meharry Teaching/Datasets/arXiv/arxiv_grqc_hepth_2024.csv')
FAISS_DIR       = ('C:/Users/Graham West/Python Notebooks/'
                   'Meharry Teaching/Datasets/arXiv/faiss_specter_grqc_hepth_2024')
CHUNK_SIZE      = 50_000
MIN_SUBMIT_YYMM = '2401'   # arXiv ID prefix: 2401 = Jan 2024, 2412 = Dec 2024, etc.
MAX_PAPERS      = None      # cap total papers (e.g. 8000 to speed up first run); None = all

# Quantum gravity research lives in both gr-qc (LQG, canonical, causal-set) and
# hep-th (string theory, holography, AdS/CFT) — include both for a complete picture.
ARCHIVES  = ['gr-qc', 'hep-th']
KEY2TOPIC = {
    'gr-qc':  'General Relativity & Quantum Cosmology',
    'hep-th': 'High Energy Physics – Theory',
}


def primary_archive(categories_str):
    """Return the top-level arXiv archive code from a space-separated category string.
    Examples: 'gr-qc astro-ph.CO' → 'gr-qc'    'hep-th' → 'hep-th'
    """
    first = str(categories_str).split()[0]
    return first.split('.')[0] if '.' in first else first


def submit_yymm(arxiv_id):
    """Extract the YYMM submission prefix from a new-style arXiv ID.

    '2401.12345' → '2401'  (submitted Jan 2024)
    '2412.99999' → '2412'  (submitted Dec 2024)
    'gr-qc/0601001' → None (old-style ID — excluded so only 2024+ papers are kept)

    Why use the ID rather than update_date? update_date reflects the last metadata edit,
    not when the paper was submitted — a 2003 paper revised in 2024 would pass an
    update_date filter but is not "new" research.
    """
    p = str(arxiv_id).split('.')[0]
    return p if (p.isdigit() and len(p) == 4) else None


def build_csv():
    """Stream the full JSON snapshot, filter to recent gr-qc + hep-th papers, write CSV."""
    keep_cols = ['id', 'title', 'abstract', 'categories', 'update_date']
    rows      = []

    print(f"Scanning: {ARXIV_PATH}")
    print(f"Keeping : archives={ARCHIVES}, submitted >= {MIN_SUBMIT_YYMM}")
    print("One-time step — saves a CSV so future runs load instantly.\n")

    n_chunks = 0
    for chunk in pd.read_json(ARXIV_PATH, lines=True, chunksize=CHUNK_SIZE):
        n_chunks += 1
        chunk           = chunk[keep_cols].copy()
        chunk['archive'] = chunk['categories'].map(primary_archive)
        chunk           = chunk[chunk['archive'].isin(ARCHIVES)]
        chunk['yymm']   = chunk['id'].map(submit_yymm)
        chunk           = chunk[chunk['yymm'].notna() & (chunk['yymm'] >= MIN_SUBMIT_YYMM)]
        word_count      = chunk['abstract'].fillna('').str.split().str.len()
        chunk           = chunk[(word_count >= 50) & (word_count <= 300)]
        rows.append(chunk)
        if n_chunks % 20 == 0:
            n_so_far = sum(len(r) for r in rows)
            print(f"  chunk {n_chunks:>4}: {n_so_far} matching papers found so far ...")

    df_out = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=keep_cols)
    df_out = df_out.sort_values('yymm', ascending=False)
    if MAX_PAPERS is not None:
        df_out = df_out.head(MAX_PAPERS)
    df_out.to_csv(ARXIV_CSV, index=False)
    print(f"\nSaved {len(df_out)} papers → {ARXIV_CSV}")
    print(f"Per-archive breakdown:\n{df_out['archive'].value_counts().to_string()}")
    return df_out


# ─── Load (build CSV on first run; fast reload thereafter) ────────────────────
if os.path.exists(ARXIV_CSV):
    print(f"Loading cached data from:\n  {ARXIV_CSV}\n")
    df = pd.read_csv(ARXIV_CSV)
else:
    df = build_csv()

df['topic'] = df['archive'].map(KEY2TOPIC)
print(f"Loaded {len(df)} papers")
print("\nPapers per archive:")
print(df['archive'].value_counts().to_string())
print(f"\nMost recent IDs (true submission date): {df['id'].head(5).tolist()}")

# ─── Build LangChain Documents (one per paper) ────────────────────────────────
docs = [
    Document(
        page_content=f"{row.title}\n\n{row.abstract}",
        metadata={
            "title":       row.title,
            "archive":     row.archive,
            "topic":       KEY2TOPIC.get(str(row.archive), str(row.archive)),
            "arxiv_id":    str(row.id),
            "url":         f"https://arxiv.org/abs/{row.id}",
            "update_date": row.update_date,
        }
    )
    for row in df.itertuples()
]

print(f"\nBuilt {len(docs)} LangChain Documents (one per paper)")
print(f"\nExample document:")
print(f"  Archive : {docs[0].metadata['archive']}")
print(f"  Topic   : {docs[0].metadata['topic']}")
print(f"  Title   : {docs[0].metadata['title'][:75]}")
print(f"  URL     : {docs[0].metadata['url']}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# One document per paper — no text splitter needed
chunks = docs
print(f"Using {len(chunks)} documents (one per paper — no text splitting)\n")

# SPECTER (Sentence Transformers version) is contrastively trained on 146k citing/cited
# paper pairs from the Semantic Scholar corpus.  That training signal — "papers that cite
# each other should be close; random pairs should be far" — makes it purpose-built for
# paper-level semantic similarity and retrieval.
#
# Contrast with SciBERT, which uses mean-pooling of token embeddings without any
# contrastive fine-tuning.  SciBERT's vocabulary understands scientific terminology, but
# its vectors aren't calibrated for similarity ranking — you often get loosely related
# papers at the top.  SPECTER fixes exactly that.
embeddings = HuggingFaceEmbeddings(
    model_name    = SPECTER_MODEL,
    model_kwargs  = {"device": DEVICE},
    encode_kwargs = {"normalize_embeddings": True, "batch_size": 64},
)

# Persist the FAISS index to disk — embedding ~15–20k papers takes several minutes
# on CPU.  Once saved, the index reloads in seconds on every subsequent run.
# To force a rebuild: delete FAISS_DIR and re-run this cell.
if os.path.exists(FAISS_DIR):
    print(f"Reloading FAISS index from disk ...")
    print(f"  {FAISS_DIR}")
    t0 = time.perf_counter()
    vectorstore = FAISS.load_local(
        FAISS_DIR, embeddings, allow_dangerous_deserialization=True
    )
    elapsed = time.perf_counter() - t0
    print(f"  Loaded in {elapsed:.1f}s")
else:
    print(f"Building FAISS index for {len(chunks)} papers ...")
    print(f"  SPECTER downloads ~440 MB from HuggingFace Hub on first use")
    print(f"  Device: {DEVICE}  (CPU: several minutes | GPU: ~1–2 min)")
    t0 = time.perf_counter()
    vectorstore = FAISS.from_documents(chunks, embeddings)
    elapsed = time.perf_counter() - t0
    vectorstore.save_local(FAISS_DIR)
    print(f"  Built in {elapsed:.1f}s → saved to {FAISS_DIR}")

print(f"\nFAISS index: {vectorstore.index.ntotal:,} vectors × {vectorstore.index.d} dimensions")

## 5. The Retriever

A **retriever** wraps the vector store and handles the semantic search step.
`search_type="similarity"` and `k=12` means: find the 12 papers whose SPECTER embeddings
are closest (by cosine similarity) to the query embedding.

We use **k=12** because:
1. Synthesizing a "current state of the field" answer benefits from broad coverage — more papers
   surface more themes, researchers, and approaches.
2. Our context window is `num_ctx=16384`, so 12 abstracts (~250 words each ≈ 3–4k tokens)
   fit comfortably with room for the prompt and the generated answer.

You can call `.invoke()` on the retriever directly to inspect what it finds — this
is useful for debugging why a RAG answer is good or bad.

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": RETRIEVE_K})

# Demonstrate retrieval directly — great for debugging why a RAG answer is good or bad
test_query = "approaches to quantizing gravity"
retrieved  = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"\nTop {len(retrieved)} retrieved papers:")
for i, doc in enumerate(retrieved, 1):
    meta = doc.metadata
    print(f"\n--- Paper {i} [{meta.get('topic', '?')}] ---")
    print(f"  Title : {meta.get('title', 'N/A')[:75]}")
    print(f"  URL   : {meta.get('url', 'N/A')}")
    print(f"  Date  : {meta.get('update_date', 'N/A')}")

## 6. Building the RAG Chain (LCEL Style)

The RAG chain works like this:

```
User question
      ↓
  Retriever  →  fetches top-12 most similar arXiv abstracts (SPECTER + FAISS)
      ↓
  format_docs  →  prefixes each abstract with its [Topic] and title
      ↓
  Prompt Template  →  inserts formatted abstracts as "context" + original question
      ↓
  LLM (Ollama gemma3:4b, num_ctx=16384)  →  synthesizes an answer grounded in retrieved papers
      ↓
  String Parser  →  returns clean text
```

`RunnablePassthrough()` passes the original question through unchanged
so it's available both for retrieval AND for the final prompt.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a scientific assistant helping to synthesize recent research literature. "
     "Answer the question using ONLY the provided arXiv abstracts. "
     "In your answer:\n"
     "  • Synthesize themes and approaches mentioned across the papers\n"
     "  • Name specific paper titles when citing findings\n"
     "  • If the abstracts don't contain the answer, say "
     "'I don\\'t have that in my knowledge base.'\n"
     "  • Do NOT use knowledge from outside the provided abstracts.\n\n"
     "Context (recent arXiv abstracts):\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    """Format retrieved documents with [Topic] / title headers for the LLM context."""
    parts = []
    for doc in docs:
        topic = doc.metadata.get("topic", "?")
        title = doc.metadata.get("title", "?")
        parts.append(f"[{topic}]\nTitle: {title}\n\n{doc.page_content.strip()}")
    return "\n\n---\n\n".join(parts)

context_chain = retriever | format_docs

rag_chain = (
    {"context": context_chain, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | parser
)

print("RAG chain assembled:")
print(f"  context_chain  → retriever | format_docs  (SPECTER + FAISS, k={RETRIEVE_K})")
print("  RAG_PROMPT     → synthesis + citation system prompt")
print(f"  llm            → ChatOllama ({LLM_MODEL}, num_ctx={NUM_CTX})")
print("  parser         → StrOutputParser")

## 7. The "Wow Moment" — RAG vs. No RAG

Same question. Same model. The only difference is whether the LLM
can see retrieved arXiv abstracts before answering.

The contrast here is not primarily about hallucination — the LLM already *knows* about
quantum gravity from training. The contrast is about **specificity, currency, and grounding**:

- **No RAG** → a generic, textbook-style overview that could have been written in 2015
- **With RAG** → a synthesized answer that names specific papers from your corpus,
  reflects recent (post-2021) approaches, and attributes claims to real arXiv preprints

This is the core value proposition of RAG for **research synthesis** tasks.

In [13]:
print("QUESTION:", KEY_QUESTION)
print()

print("=" * 65)
print("WITHOUT RAG  (LLM answering from training weights only)")
print("=" * 65)
print(textwrap.fill(answer_no_rag, width=80))

print()
print("=" * 65)
print("WITH RAG  (LLM grounded in recent arXiv physics abstracts)")
print("=" * 65)
t0 = time.perf_counter()
answer_with_rag = rag_chain.invoke(KEY_QUESTION)
elapsed = time.perf_counter() - t0
print(textwrap.fill(answer_with_rag, width=80))
print(f"\n[{elapsed:.1f}s]")

# Show which papers were retrieved and are available to the LLM
retrieved_docs = retriever.invoke(KEY_QUESTION)
print()
print("─" * 65)
print("PAPERS RETRIEVED AND AVAILABLE TO THE LLM:")
for doc in retrieved_docs:
    meta = doc.metadata
    print(f"  [{meta.get('topic','?')[:35]}]")
    print(f"  {meta.get('title','?')[:70]}")
    print(f"  {meta.get('url','?')}")
    print()
print("Notice: the RAG answer names specific papers and reflects the corpus.")
print("The no-RAG answer is generic and cannot cite any source.")

QUESTION: What is the current state of quantum gravity research?

WITHOUT RAG  (LLM answering from training weights only)
Quantum gravity research is active but lacks a definitive theory. Here’s a
concise overview:  *   **String Theory:** Remains the most prominent framework,
exploring gravity as arising from vibrating strings. *   **Loop Quantum
Gravity:** Focuses on quantizing spacetime itself, proposing it’s made of
discrete loops. *   **Other Approaches:** Include causal set theory, asymptotic
safety, and emergent gravity. *   **Experimental Challenges:** Direct
experimental verification is extremely difficult due to the extremely small
scales involved. Researchers are exploring potential indirect signatures like
subtle effects on cosmology or black holes.  Essentially, it’s a field of
ongoing theoretical investigation with no established consensus.

WITH RAG  (LLM grounded in recent arXiv physics abstracts)
Based solely on the provided arXiv abstracts, here’s a synthesis of the cu

## 8. Quality Check — Gemini + Google Search as an Independent Judge

Can we *objectively* test whether RAG improved the answer?

One approach: ask a **different LLM from a different provider, with live web access**,
to fact-check both answers against the real literature.

**Gemini 2.5 Flash** with **Google Search grounding** can retrieve current arXiv coverage
and assess whether each answer is:
- **Accurate** — does it match what the research community actually works on?
- **Current** — does it reflect recent (post-2021) advances, or a generic overview?
- **Grounded** — does it name specific papers and approaches, or make vague claims?

This pattern is called **LLM-as-judge** and is widely used in evaluation and alignment
research. Using a *different* provider (Google vs. local Ollama) reduces the risk that
the judge is systematically biased toward its own output style.

> **API key required:** `GEMINI_API_KEY` — set as an environment variable.  
> **Note:** Google Search grounding cannot be combined with JSON `response_schema` in the
> same API call, so the judge returns formatted free text rather than a structured object.

In [14]:
import os

try:
    from google import genai
    from google.genai import types as gtypes

    api_key = os.environ.get("GEMINI_API_KEY", "")
    if not api_key:
        print("⚠  GEMINI_API_KEY not set — skipping judge step.")
        print("   Set it with:  import os; os.environ['GEMINI_API_KEY'] = 'your-key'")
    else:
        client = genai.Client(api_key=api_key)

        JUDGE_RUBRIC = """\
You are an expert in fundamental physics evaluating an answer to the question:

"{question}"

Answer to evaluate ({label}):
{answer}

Using web search to check against current arXiv literature, assess this answer on:
1. **Accuracy** (1–5): Are the claims factually correct per the recent physics literature?
2. **Currency** (1–5): Does it reflect post-2021 research, or a generic textbook-level overview?
3. **Grounding** (1–5): Does it name specific papers, authors, or approaches — or is it vague?

Briefly justify each score (1–2 sentences per dimension).
Flag any specific errors or outdated claims.
List the web sources you consulted under **Sources:**.
"""

        def judge_answer(question: str, answer: str, label: str) -> str:
            prompt = JUDGE_RUBRIC.format(question=question, label=label, answer=answer)
            resp = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=gtypes.GenerateContentConfig(
                    tools=[gtypes.Tool(google_search=gtypes.GoogleSearch())],
                    temperature=0,
                ),
            )
            return resp.text

        print("=" * 65)
        print("GEMINI JUDGE — WITHOUT RAG")
        print("=" * 65)
        verdict_no_rag = judge_answer(KEY_QUESTION, answer_no_rag, "WITHOUT RAG")
        print(verdict_no_rag)

        print()
        print("=" * 65)
        print("GEMINI JUDGE — WITH RAG")
        print("=" * 65)
        verdict_with_rag = judge_answer(KEY_QUESTION, answer_with_rag, "WITH RAG")
        print(verdict_with_rag)

        print()
        print("Do the RAG scores improve on accuracy and grounding?")
        print("The web-sourced judge gives us an external, non-Ollama reference point.")

except ImportError:
    print("⚠  google-genai not installed. Run:  pip install google-genai")
except Exception as e:
    print(f"⚠  Gemini judge failed: {type(e).__name__}: {e}")

GEMINI JUDGE — WITHOUT RAG
Here's an evaluation of the provided answer on the current state of quantum gravity research:

### Evaluation

1.  **Accuracy** (5/5)
    The claims made are factually correct. Quantum gravity research is indeed active and lacks a definitive theory. The descriptions of String Theory, Loop Quantum Gravity, and the mention of other approaches like causal set theory, asymptotic safety, and emergent gravity accurately reflect the major research directions. The statement about experimental challenges and the exploration of indirect signatures is also correct.

2.  **Currency** (4/5)
    The answer provides a current overview of the field. The general descriptions of the main approaches remain relevant in post-2021 literature. Recent arXiv preprints from 2024, 2025, and 2026 confirm the ongoing activity and the nature of research in string theory, loop quantum gravity, causal set theory, asymptotic safety, and emergent gravity, as well as the focus on indirect expe

## 9. Conversational Memory — Explicit Pattern

This rewrite avoids `langchain.chains` entirely. Each step is explicit Python:

1. **Contextualize** — if there is chat history, call the LLM to rewrite a vague follow-up as a standalone question before it hits the retriever
2. **Retrieve** — call `retriever.invoke(standalone_question)` directly
3. **Prompt** — format a `ChatPromptTemplate` with the retrieved context, chat history, and question
4. **Call LLM** — `llm.invoke(messages)` returns an `AIMessage`
5. **Record** — append `HumanMessage` / `AIMessage` to the plain `chat_history` list

**Why contextualization matters:**
A vague follow-up like *"And how do they connect to holography?"*
embeds poorly on its own — the retriever may return off-topic papers. Rewriting it first
produces something like *"How do quantum gravity approaches relate to the holographic principle?"*
which retrieves the right abstracts.

**Chat history is just a list:** no memory objects or session stores — a plain
`list[HumanMessage | AIMessage]` passed explicitly on every call.

In [15]:
import textwrap, time
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# --- Prompt: rewrite a follow-up as a self-contained standalone question ---
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, rewrite it as a "
     "standalone question that is fully self-contained. "
     "Return ONLY the rewritten question, nothing else."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
contextualize_chain = contextualize_prompt | llm | parser

# --- Prompt: answer using retrieved context + history ---
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a scientific assistant. Answer using ONLY the provided arXiv abstracts. "
     "Be specific — name paper titles when citing findings.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Chat history is a plain Python list — no memory objects needed
chat_history = []

def format_docs_with_titles(docs):
    parts = []
    for doc in docs:
        topic = doc.metadata.get("topic", "?")
        title = doc.metadata.get("title", "?")
        parts.append(f"[{topic}]\nTitle: {title}\n\n{doc.page_content.strip()}")
    return "\n\n---\n\n".join(parts)

def ask(question: str) -> str:
    t0 = time.perf_counter()

    # Rewrite vague follow-ups so retrieval stays accurate across turns
    standalone = (
        contextualize_chain.invoke({"input": question, "chat_history": chat_history})
        if chat_history else question
    )

    docs = retriever.invoke(standalone)
    context = format_docs_with_titles(docs)

    messages = qa_prompt.format_messages(
        context=context,
        chat_history=chat_history,
        input=question,
    )
    answer = llm.invoke(messages).content

    chat_history.extend([HumanMessage(content=question), AIMessage(content=answer)])

    elapsed = time.perf_counter() - t0
    print(f"Q: {question}")
    print(f"A: {textwrap.fill(answer, width=75)}")
    print(f"   [{len(docs)} papers retrieved | {elapsed:.1f}s]\n")
    return answer

# Multi-turn conversation — the third question requires history to resolve correctly
_ = ask("What is the current state of quantum gravity research?")
_ = ask("Which of those approaches involve string theory?")
_ = ask("And how do they connect to black holes or holography?")

Q: What is the current state of quantum gravity research?
A: Based solely on the provided arXiv abstracts, here’s a snapshot of the
current state of quantum gravity research, broken down by the areas
represented:  *   **Dark Sector Mediators & Angular Distributions:**
Research is focused on using experimental data from facilities like DUNE,
SHiP, and FASER2 to determine the spin of dark sector mediators. The key
idea is to analyze the angular distributions of decay products (vector
bosons and scalars) to infer the mediator’s quantum numbers.  *   **Black
Hole Information Paradox:** The “magic of the gravitational vacuum” paper
explores the Vecro hypothesis, which proposes a structure for the
gravitational vacuum that could potentially resolve the black hole
information paradox by “feeling around” closed trapped surfaces and
nucleating fuzzball structures. This involves a lattice model with
extended-scale correlations.  *   **Newtonian Gravity Extensions:**
Research continues on extendi

## 10. Inspecting Memory State

You can always inspect what the memory object currently holds.
This shows exactly what gets prepended to the LLM's context on each new turn.

In [16]:
print("Current conversation history:")
print("-" * 50)
for msg in chat_history:
    role = "Human" if msg.type == "human" else "AI"
    print(f"[{role}]\n{textwrap.fill(msg.content, width=75)}\n")

Current conversation history:
--------------------------------------------------
[Human]
What is the current state of quantum gravity research?

[AI]
Based solely on the provided arXiv abstracts, here’s a snapshot of the
current state of quantum gravity research, broken down by the areas
represented:  *   **Dark Sector Mediators & Angular Distributions:**
Research is focused on using experimental data from facilities like DUNE,
SHiP, and FASER2 to determine the spin of dark sector mediators. The key
idea is to analyze the angular distributions of decay products (vector
bosons and scalars) to infer the mediator’s quantum numbers.  *   **Black
Hole Information Paradox:** The “magic of the gravitational vacuum” paper
explores the Vecro hypothesis, which proposes a structure for the
gravitational vacuum that could potentially resolve the black hole
information paradox by “feeling around” closed trapped surfaces and
nucleating fuzzball structures. This involves a lattice model with
extended

## 11. Try Your Own Question

Use the cell below to ask anything about the physics abstracts in the knowledge base.
Retrieval is limited to the ~15–20k gr-qc and hep-th papers in the FAISS index — the LLM
will say so if the answer isn't covered.

To reset memory for a fresh conversation:
```python
chat_history.clear()
```

Ideas to try:
- *"What are the main mathematical tools used in loop quantum gravity?"*
- *"Summarize recent work on the black hole information paradox."*
- *"Which papers discuss AdS/CFT or the holographic principle?"*
- *"What is the current status of causal set theory?"*
- *"How are observational constraints being applied to quantum gravity theories?"*

In [17]:
# Reset for a fresh conversation
chat_history.clear()

my_question = "What approaches to quantum gravity have gained the most traction since 2021?"
_ = ask(my_question)

Q: What approaches to quantum gravity have gained the most traction since 2021?
A: Based solely on the provided arXiv abstracts, here’s a breakdown of
approaches to quantum gravity that have gained traction since 2021, as
indicated by the number of mentions and the detail of the work:  1.
**Asymptotic Safety:** The paper “The pole truth: an analytical graviton
propagator from Asymptotic Safety” (2023) highlights significant work in
this area. The abstract indicates a focus on deriving an analytical
approximation for the graviton propagator, aiming to demonstrate that
Asymptotic Safety doesn’t introduce new degrees of freedom or violate
unitarity/causality.  2.  **Vecro Hypothesis:** The paper “The magic of the
gravitational vacuum” (2023) presents the vecro hypothesis as a potential
solution to the black hole information paradox. This approach is receiving
attention due to its attempt to circumvent the semiclassical approximation.
3.  **Primordial Parity Violation & Galaxy Spins:** The

## 12. Summary — What LangChain Added vs Plain `import ollama`

| Capability | Direct `ollama` | LangChain |
|---|---|---|
| Basic chat | `ollama.chat(...)` | `ChatOllama` + LCEL chain |
| Prompt templating | f-strings | `ChatPromptTemplate` |
| Multi-step pipelines | nested functions | `\|` pipe operator (LCEL) |
| Semantic retrieval | manual cosine math | `FAISS.as_retriever()` |
| Conversation history | manual list management | Manual `chat_history` list (explicit pattern) |
| Full RAG pipeline | ~50 lines + custom logic | LCEL `rag_chain` (5 lines) |
| Swap backends | rewrite everything | change one class name |

*Note: this notebook uses an **explicit manual memory pattern** — a plain `list[HumanMessage | AIMessage]`
passed explicitly on every call — rather than LangChain's `ConversationBufferMemory` object.
Both approaches work; the explicit pattern is more transparent for teaching and easier to debug.*

**When to use direct `ollama`:** Single-turn calls, minimal dependencies, maximum transparency.

**When to use LangChain:** Multi-step pipelines, RAG, memory, when components need to be swappable.

---

### Why the RAG Answer Improved (Three Compounding Fixes)

The earlier run with 600 mixed-archive papers produced a RAG answer *worse* than no-RAG.
Three changes fixed it together — each alone would not have been enough:

1. **Focused corpus (gr-qc + hep-th, 2024+):** Quantum gravity research is not spread evenly
   across physics — concentrating on the two archives where it lives dramatically improves the
   odds that a top-k retrieval is *relevant*.
2. **SPECTER instead of SciBERT:** SPECTER is contrastively trained for paper-level similarity;
   SciBERT uses mean-pooling without contrastive fine-tuning. The embedder choice matters as
   much as corpus size.
3. **Larger Ollama context (`num_ctx=16384`):** Ollama's default context is ~2048 tokens.
   With k=12 retrieved abstracts (~3–4k tokens of context), everything was silently truncated
   before the model ever saw it. Enlarging `num_ctx` lets the full retrieved context reach
   the LLM — the most direct fix.

### Next Steps
- Swap archives: change `ARCHIVES` to e.g. `['cs.LG', 'stat.ML']` for an ML research demo
- Increase `RETRIEVE_K` to 20 and observe diminishing returns vs. latency
- Try a reranker between retriever and LLM (cross-encoder) for better chunk selection
- Extend `judge_answer()` to evaluate the multi-turn conversation turns
- Stream responses token-by-token: `for chunk in rag_chain.stream(question): print(chunk, end="")`
- Swap FAISS for Chroma: replace `FAISS.from_documents(...)` with `Chroma.from_documents(...)`
- Try SPECTER2 (`allenai/specter2`) for even better retrieval on recent literature